<a href="https://www.kaggle.com/code/avikdas567/march-machine-learning-mania-2026?scriptVersionId=299074806" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/march-machine-learning-mania-2026/Conferences.csv
/kaggle/input/march-machine-learning-mania-2026/WNCAATourneyDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/MNCAATourneySeedRoundSlots.csv
/kaggle/input/march-machine-learning-mania-2026/MRegularSeasonDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/MGameCities.csv
/kaggle/input/march-machine-learning-mania-2026/WSecondaryTourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/WGameCities.csv
/kaggle/input/march-machine-learning-mania-2026/MSeasons.csv
/kaggle/input/march-machine-learning-mania-2026/WNCAATourneySlots.csv
/kaggle/input/march-machine-learning-mania-2026/MSecondaryTourneyTeams.csv
/kaggle/input/march-machine-learning-mania-2026/SampleSubmissionStage2.csv
/kaggle/input/march-machine-learning-mania

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/input/march-machine-learning-mania-2026"


# Build team-season ratings

def build_team_ratings(prefix):
    df = pd.read_csv(f"{DATA_PATH}/{prefix}RegularSeasonCompactResults.csv")

    win = df[["Season","WTeamID","WScore","LScore"]].copy()
    win.columns = ["Season","TeamID","ScoreFor","ScoreAgainst"]
    win["Win"] = 1

    lose = df[["Season","LTeamID","LScore","WScore"]].copy()
    lose.columns = ["Season","TeamID","ScoreFor","ScoreAgainst"]
    lose["Win"] = 0

    games = pd.concat([win, lose], ignore_index=True)
    games["Margin"] = games["ScoreFor"] - games["ScoreAgainst"]

    ratings = games.groupby(["Season","TeamID"]).agg(
        games=("Win","count"),
        win_rate=("Win","mean"),
        avg_margin=("Margin","mean")
    ).reset_index()

    return ratings


# Prepare training data

def build_training_data(prefix):
    results = pd.read_csv(f"{DATA_PATH}/{prefix}RegularSeasonCompactResults.csv")
    ratings = build_team_ratings(prefix)

    df = results.merge(
        ratings, left_on=["Season","WTeamID"], right_on=["Season","TeamID"]
    ).merge(
        ratings, left_on=["Season","LTeamID"], right_on=["Season","TeamID"],
        suffixes=("_W","_L")
    )

    X = pd.DataFrame({
        "win_rate_diff": df["win_rate_W"] - df["win_rate_L"],
        "margin_diff": df["avg_margin_W"] - df["avg_margin_L"],
    })

    y = np.ones(len(df))
    seasons = df["Season"]

    # Add reverse games
    X_rev = -X
    y_rev = np.zeros(len(df))

    X = pd.concat([X, X_rev], ignore_index=True)
    y = np.concatenate([y, y_rev])
    seasons = pd.concat([seasons, seasons], ignore_index=True)

    return X, y, seasons, ratings


# Load MEN + WOMEN

X_m, y_m, s_m, ratings_m = build_training_data("M")
X_w, y_w, s_w, ratings_w = build_training_data("W")

X = pd.concat([X_m, X_w], ignore_index=True)
y = np.concatenate([y_m, y_w])
seasons = pd.concat([s_m, s_w], ignore_index=True)

mask = seasons.between(2022, 2025)
X_train = X[mask]
y_train = y[mask]


# Train model

model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)


# Submission

sub = pd.read_csv(f"{DATA_PATH}/SampleSubmissionStage1.csv")
ids = sub["ID"].str.split("_", expand=True)
sub["Season"] = ids[0].astype(int)
sub["Team1"] = ids[1].astype(int)
sub["Team2"] = ids[2].astype(int)

def predict(sub_df, ratings):
    df = sub_df.merge(
        ratings, left_on=["Season","Team1"], right_on=["Season","TeamID"], how="left"
    ).merge(
        ratings, left_on=["Season","Team2"], right_on=["Season","TeamID"],
        suffixes=("_1","_2"), how="left"
    )

    df["win_rate_1"] = df["win_rate_1"].fillna(0.5)
    df["win_rate_2"] = df["win_rate_2"].fillna(0.5)
    df["avg_margin_1"] = df["avg_margin_1"].fillna(0)
    df["avg_margin_2"] = df["avg_margin_2"].fillna(0)

    Xp = pd.DataFrame({
        "win_rate_diff": df["win_rate_1"] - df["win_rate_2"],
        "margin_diff": df["avg_margin_1"] - df["avg_margin_2"]
    })

    return model.predict_proba(Xp)[:,1]

men_mask = sub["Team1"] < 2000
women_mask = sub["Team1"] >= 3000

sub.loc[men_mask, "Pred"] = predict(sub[men_mask], ratings_m)
sub.loc[women_mask, "Pred"] = predict(sub[women_mask], ratings_w)

sub["Pred"] = sub["Pred"].clip(0.01, 0.99)

submission = sub[["ID","Pred"]]
assert submission.shape[0] == 519144
submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully")
submission.head()

submission.csv created successfully


,ID,Pred
0,2022_1101_1102,0.861052
1,2022_1101_1103,0.438105
2,2022_1101_1104,0.580522
3,2022_1101_1105,0.839667
4,2022_1101_1106,0.894025
